# ⚡ Concurrent Agent Workflows wit Microsoft Foundry (Python)

## 📋 Advanced Parallel Processing Tutorial

Dis notebook dey show **concurrent workflow patterns** wey dey use Microsoft Agent Framework. You go learn how to build high-performance, parallel processing workflows wey multiple AI agents dey run at di same time, wey go improve throughput well well and fit run correct multi-threaded business processes.

> **Migration note:** Dis sample before e dey use GitHub Models. GitHub Models done retire (e go stop July 2026), so now e dey use **Microsoft Foundry** through di `FoundryChatClient`, wey dey target di Azure OpenAI **Responses API**.

## 🎯 Wetin You Go Learn

### 🚀 **Concurrent Processing Basics**
- **Parallel Agent Execution**: Make many agents dey run at di same time to get max efficiency
- **Workflow Orchestration**: Arrange concurrent operations make data still dey correct
- **Performance Optimization**: Get plenty speedup wit parallel processing
- **Resource Management**: Use AI model resources well well for concurrent operations

### 🏗️ **Advanced Concurrency Patterns**
- **Fork-Join Processing**: Divide work for many agents then join results together
- **Pipeline Parallelism**: Make execution stages dey overlap for steady throughput
- **Load Balancing**: Share work balance among available agent resources
- **Synchronization Points**: Arrange concurrent agents for important workflow stages

### 🏢 **Enterprise Concurrent Applications**
- **High-Volume Document Processing**: Process many documents at the same time
- **Real-Time Content Analysis**: Concurrently analyse incoming data streams
- **Batch Processing Optimization**: Maximize throughput for big operations
- **Multi-Modal Analysis**: Parallel processing of different content types (text, images, data)

## ⚙️ Wetin You Need & How to Set Up

### 📦 **Wetin You Must Install**

Install Agent Framework wey get concurrent workflow abilities:

```bash
pip install agent-framework -U
```

### 🔑 **Microsoft Foundry Configuration**

Sign in wit Azure CLI (`az login`) so `AzureCliCredential` fit authenticate, then set your Microsoft Foundry project details.

**Environment Setup (.env file):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

**Concurrent Processing Tings to Note:**
- **Rate Limits**: Check Azure OpenAI rate limits for concurrent requests
- **Resource Usage**: Think about memory and CPU use when multiple agents dey run
- **Error Handling**: Make correct error recovery for parallel operations

### 🏗️ **Concurrent Workflow Architecture**

```mermaid
graph TD
    A[Workflow Don Start] --> B[Dem Dey Run Together]
    B --> C[Agent Pool 1]
    B --> D[Agent Pool 2]
    B --> E[Agent Pool 3]
    C --> F[Result Dem Join Body]
    D --> F
    E --> F
    F --> G[Final Output]
    
    H[Microsoft Foundry] --> C
    H --> D
    H --> E
```

**Main Benefits:**
- **⚡ Performance**: Big speedup wit parallel execution
- **📈 Scalability**: Fit handle more work without time increase
- **🔄 Efficiency**: Better use of available computational resources
- **🎯 Throughput**: Process more work in di same time

## 🎨 **Concurrent Workflow Design Patterns**

### 🔍 **Research & Analysis Pipeline**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Data Processing Workflow**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Content Creation Pipeline**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Multi-Stage Processing**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Enterprise Performance Benefits**

### ⚡ **Throughput Optimization**
- **Parallel Execution**: Many agents dey work together at di same time
- **Resource Utilization**: Maximum efficiency for available AI model capacity
- **Time Reduction**: Plenty reduction for total processing time
- **Scalable Architecture**: Easy to add more concurrent agents as needed

### 🛡️ **Reliability & Resilience**
- **Fault Tolerance**: One agent failure no go stop the whole workflow
- **Error Isolation**: Problem for one concurrent branch no go affect the others
- **Graceful Degradation**: System go still dey work even if agent capacity reduce
- **Recovery Mechanisms**: Automatic retry and error handling for operations wey fail

### 📊 **Monitoring & Observability**
- **Concurrent Execution Tracking**: Track progress of all parallel operations
- **Performance Metrics**: Measure how speed and efficiency improve
- **Resource Usage Analytics**: Make better allocation of concurrent agents
- **Bottleneck Identification**: Find and fix performance wahala

Make we build high-performance concurrent AI workflows! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = provider.as_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = provider.as_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)


In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
